# Starter notebook · From data to a Kaggle submission

**Machine Learning I · Group Project 2026-27**

This notebook is a *taxi to the runway*, not a flight. It shows the mechanics end to end:

1. load `learn.csv` and `predict.csv`;
2. train a **dummy classifier** (it ignores every feature);
3. estimate its F1 score using the holdout method;
4. write a submission file in the format Kaggle expects.

A dummy model is the score your real model has to beat. If it doesn't, something is wrong.

In [1]:
import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score

GROUP = "00"          # group number, two digits
SEED = 5


## 1. Load the data

In [2]:
learn = pd.read_csv("learn.csv")
predict = pd.read_csv("predict.csv")

print(f"learn:   {learn.shape[0]:,} rows x {learn.shape[1]} columns")
print(f"predict: {predict.shape[0]:,} rows x {predict.shape[1]} columns")
print(f"fatal rate in learn: {learn['fatal'].mean():.1%}")

learn:   16,979 rows x 42 columns
predict: 4,245 rows x 41 columns
fatal rate in learn: 17.0%


In [3]:
X = learn.drop(columns=["row_id", "fatal"])
y = learn["fatal"]
X_predict = predict.drop(columns=["row_id"])


In [4]:
X_train, X_val, y_train, y_val = train_test_split( X, y, random_state=5, stratify=y)

## 2. Multiple dummy strategies

A `DummyClassifier` looks only at the target, never at the features:

| Strategy | Predicts |
|---|---|
| `most_frequent` | always 0 (the most common class) |
| `stratified` | 0 or 1 at random, in the same proportion as y_train |
| `constant` (=1) | always 1 |
| `uniform` | predict 50% 0 and 50% 1 |

The project metric is **F1 on the fatal class (1)**. Watch what each strategy scores.

In [5]:

strategies = {
    "always 0": DummyClassifier(strategy="most_frequent"),
    "always 1": DummyClassifier(strategy="constant", constant=1),
    "random, same proportion": DummyClassifier(strategy="stratified", random_state=SEED),
    "random, uniform": DummyClassifier(strategy="uniform", random_state=SEED)
    }



results = {}
for name, model in strategies.items():
    model.fit( X_train, y_train)
    pred_val = model.predict(X_val)
    resmetrics = {}
    resmetrics["F1"] = f1_score( y_val, pred_val, average="binary")
    resmetrics["Precision"] = precision_score( y_val, pred_val, zero_division=0.0)
    resmetrics["Recall"] = recall_score( y_val, pred_val)
    resmetrics["Accuracy"] = accuracy_score( y_val, pred_val)
    results[name] = resmetrics

pd.DataFrame(results).T.round(3)


,F1,Precision,Recall,Accuracy
always 0,0.000,0.000,0.000,0.830
always 1,0.291,0.170,1.000,0.170
"random, same proportion",0.183,0.188,0.179,0.729
"random, uniform",0.260,0.174,0.514,0.502


**Read this table carefully.** "Always 0" is right 83% of the time, yet its F1 is 0:
it never finds a single fatal accident. "Always 1" finds all of them and reaches about 0.29.
Accuracy would tell you the opposite story, which is exactly why it isn't the project metric.

## 3. Train on all of `learn.csv` and predict

In [6]:
model = DummyClassifier(strategy="most_frequent")
model.fit(X, y)
predictions = model.predict(X_predict)


## 4. Write the submission

Kaggle expects exactly two columns, `row_id` and `fatal`, with one row per row of `predict.csv`.

In [7]:
submission = pd.DataFrame({"row_id": predict["row_id"], "fatal": predictions.astype(int)})

assert len(submission) == len(predict)
assert submission["fatal"].isin([0, 1]).all()

file_name = f"ML{GROUP}_dummy_submission.csv"
submission.to_csv(file_name, index=False)
print(f"wrote {file_name}")
submission.head()


wrote ML00_dummy_submission.csv


,row_id,fatal
0,A00003,0
1,A00009,0
2,A00023,0
3,A00025,0
4,A00034,0


Upload the file on the competition page (**Submit Prediction**). Your public score should be close
to the F1 score above.

**Now go and build something that flies.**